# 02. Classical Models

This notebook is a small, presentation-friendly classical forecasting study built only on the 4 representative series selected in `01_EDA.ipynb`:

- longest history
- highest total weight
- most volatile
- most stable

The goal here is simple: compare a few non-trivial classical models on four very different series and look at the forecast behavior clearly.


In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import Holt, SimpleExpSmoothing

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "ts-forecasting"
VAL_CUTOFF = 2880
SERIES_KEYS = ["code", "sub_code", "sub_category", "horizon"]
REASON_ORDER = ["longest history", "highest total weight", "most volatile", "most stable"]

train = pd.read_parquet(
    DATA_DIR / "train.parquet",
    columns=["code", "sub_code", "sub_category", "horizon", "ts_index", "y_target", "weight"],
)
train = train.sort_values(SERIES_KEYS + ["ts_index"]).reset_index(drop=True)
series_grouped = train.groupby(SERIES_KEYS, sort=False)

print(f"Train rows: {len(train):,}")
print(f"Validation cutoff: {VAL_CUTOFF}")


Train rows: 5,337,414
Validation cutoff: 2880


## 1. Recreate the 4 representative series

This uses the same logic as `01_EDA.ipynb`: only series that cross the cutoff, length at least 120, then pick one each for longest history, highest total weight, most volatile, and most stable.


In [2]:
series_stats = (
    train.groupby(SERIES_KEYS)
    .agg(
        length=("ts_index", "size"),
        start=("ts_index", "min"),
        end=("ts_index", "max"),
        total_weight=("weight", "sum"),
        target_std=("y_target", "std"),
    )
    .reset_index()
)
series_stats["crosses_cutoff"] = (series_stats["start"] <= VAL_CUTOFF) & (series_stats["end"] > VAL_CUTOFF)

eligible = series_stats[(series_stats["crosses_cutoff"]) & (series_stats["length"] >= 120)].copy()
eligible["target_std"] = eligible["target_std"].fillna(0.0)
stable_pool = eligible[eligible["target_std"] > 0].copy()

chosen = pd.concat(
    [
        eligible.nlargest(1, "length").assign(reason="longest history"),
        eligible.nlargest(1, "total_weight").assign(reason="highest total weight"),
        eligible.nlargest(1, "target_std").assign(reason="most volatile"),
        stable_pool.nsmallest(1, "target_std").assign(reason="most stable"),
    ],
    ignore_index=True,
).drop_duplicates(subset=SERIES_KEYS)

chosen["reason"] = pd.Categorical(chosen["reason"], categories=REASON_ORDER, ordered=True)
chosen = chosen.sort_values("reason").reset_index(drop=True)
chosen["series_label"] = chosen["reason"].astype(str).str.title()

expected = pd.DataFrame(
    [
        {"code": "X9BZ68VQ", "sub_code": "OYJGNSQK", "sub_category": "DPPUO5X2", "horizon": 1, "reason": "longest history"},
        {"code": "SJZP0OVU", "sub_code": "OYJGNSQK", "sub_category": "NQ58FVQM", "horizon": 25, "reason": "highest total weight"},
        {"code": "W4S29LF4", "sub_code": "KL66VIS3", "sub_category": "PHHHVYZI", "horizon": 25, "reason": "most volatile"},
        {"code": "SJZP0OVU", "sub_code": "OYJGNSQK", "sub_category": "NQ58FVQM", "horizon": 1, "reason": "most stable"},
    ]
)
expected["reason"] = pd.Categorical(expected["reason"], categories=REASON_ORDER, ordered=True)
expected = expected.sort_values("reason").reset_index(drop=True)
pd.testing.assert_frame_equal(chosen[[*SERIES_KEYS, "reason"]].reset_index(drop=True), expected, check_dtype=False)

display(chosen[["series_label", *SERIES_KEYS, "length", "total_weight", "target_std"]].round({"total_weight": 2, "target_std": 4}))


,series_label,code,sub_code,sub_category,horizon,length,total_weight,target_std
0,Longest History,X9BZ68VQ,OYJGNSQK,DPPUO5X2,1,212,9.060000e+02,2.6416
1,Highest Total Weight,SJZP0OVU,OYJGNSQK,NQ58FVQM,25,156,4.349747e+10,0.0004
2,Most Volatile,W4S29LF4,KL66VIS3,PHHHVYZI,25,162,4.600000e-01,296.7600
3,Most Stable,SJZP0OVU,OYJGNSQK,NQ58FVQM,1,170,3.869062e+10,0.0001


## 2. Raw series overview

These four series are intentionally very different. That is useful, but it also means some of them have extremely short pre-cutoff training windows.


In [3]:
def get_series(row):
    key = (row["code"], row["sub_code"], row["sub_category"], row["horizon"])
    return series_grouped.get_group(key)[["ts_index", "y_target", "weight"]].reset_index(drop=True)


def split_series(series_df):
    train_part = series_df[series_df["ts_index"] <= VAL_CUTOFF].copy()
    val_part = series_df[series_df["ts_index"] > VAL_CUTOFF].copy()
    return train_part, val_part


series_summary_rows = []
for _, row in chosen.iterrows():
    series_df = get_series(row)
    train_part, val_part = split_series(series_df)
    series_summary_rows.append(
        {
            "series_label": row["series_label"],
            "train_len": len(train_part),
            "val_len": len(val_part),
            "mean_y": series_df["y_target"].mean(),
            "std_y": series_df["y_target"].std(),
            "total_weight": series_df["weight"].sum(),
        }
    )

series_summary = pd.DataFrame(series_summary_rows)
display(series_summary.round({"mean_y": 4, "std_y": 4, "total_weight": 2}))

fig, axes = plt.subplots(len(chosen), 1, figsize=(13, 10), sharex=False)
axes = np.atleast_1d(axes)

for ax, (_, row) in zip(axes, chosen.iterrows()):
    series_df = get_series(row)
    ax.plot(series_df["ts_index"], series_df["y_target"], color="#4C72B0", linewidth=1.1)
    ax.axvline(VAL_CUTOFF, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{row['series_label']} - {row['code']} / {row['sub_code']} / H{row['horizon']}")
    ax.set_ylabel("y_target")

axes[-1].set_xlabel("ts_index")
plt.tight_layout()
plt.show()


,series_label,train_len,val_len,mean_y,std_y,total_weight
0,Longest History,55,157,-0.1051,2.6416,9.060000e+02
1,Highest Total Weight,7,149,-0.0001,0.0004,4.349747e+10
2,Most Volatile,131,31,15.3199,296.7600,4.600000e-01
3,Most Stable,7,163,-0.0000,0.0001,3.869062e+10


## 3. Model lineup

The trivial baselines `Zero`, `Naive`, and `Drift` are removed here. They are useful for benchmarking, but they make the forecast overlay chart look flat by construction.

The notebook now uses:

- `Rolling Mean (w=10)`
- `SES`
- `Holt`
- `ARIMA(1,0,0)`
- `ARIMA(0,1,1)`
- `ARIMA(1,1,0)`
- `ARIMA(1,1,1)`


In [4]:
def weighted_skill(y_true, y_pred, weight):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weight = np.asarray(weight, dtype=float)
    denom = np.sum(weight * (y_true ** 2))
    if denom == 0:
        return np.nan
    ratio = np.sum(weight * ((y_true - y_pred) ** 2)) / denom
    ratio = min(max(ratio, 0.0), 1.0)
    return float(np.sqrt(1.0 - ratio))


def weighted_rmse(y_true, y_pred, weight):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weight = np.asarray(weight, dtype=float)
    if np.sum(weight) == 0:
        return np.nan
    return float(np.sqrt(np.sum(weight * ((y_true - y_pred) ** 2)) / np.sum(weight)))


def weighted_mae(y_true, y_pred, weight):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weight = np.asarray(weight, dtype=float)
    if np.sum(weight) == 0:
        return np.nan
    return float(np.sum(weight * np.abs(y_true - y_pred)) / np.sum(weight))


def mase(y_true, y_pred, train_scale):
    if not np.isfinite(train_scale) or train_scale == 0:
        return np.nan
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)) / train_scale)


def forecast_rolling_mean(y, steps, window=10):
    w = min(window, len(y))
    return np.repeat(float(y.iloc[-w:].mean()), steps)


def forecast_ses(y, steps):
    fit = SimpleExpSmoothing(y, initialization_method="estimated").fit(optimized=True)
    return np.asarray(fit.forecast(steps), dtype=float)


def forecast_holt(y, steps):
    fit = Holt(y, initialization_method="estimated").fit(optimized=True)
    return np.asarray(fit.forecast(steps), dtype=float)


def make_arima_forecaster(order):
    def _forecast(y, steps):
        fit = ARIMA(
            y,
            order=order,
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit()
        pred = np.asarray(fit.forecast(steps), dtype=float)
        if not np.all(np.isfinite(pred)):
            raise ValueError("non_finite_forecast")
        return pred
    return _forecast


MODEL_SPECS = {
    "Rolling Mean (w=10)": {"min_train": 2, "func": lambda y, s: forecast_rolling_mean(y, s, 10)},
    "SES": {"min_train": 3, "func": forecast_ses},
    "Holt": {"min_train": 5, "func": forecast_holt},
    "ARIMA(1,0,0)": {"min_train": 20, "func": make_arima_forecaster((1, 0, 0))},
    "ARIMA(0,1,1)": {"min_train": 20, "func": make_arima_forecaster((0, 1, 1))},
    "ARIMA(1,1,0)": {"min_train": 20, "func": make_arima_forecaster((1, 1, 0))},
    "ARIMA(1,1,1)": {"min_train": 20, "func": make_arima_forecaster((1, 1, 1))},
}

display(pd.DataFrame({"model": list(MODEL_SPECS), "min_train": [spec["min_train"] for spec in MODEL_SPECS.values()]}))


,model,min_train
0,Rolling Mean (w=10),2
1,SES,3
2,Holt,5
3,"ARIMA(1,0,0)",20
4,"ARIMA(0,1,1)",20
5,"ARIMA(1,1,0)",20
6,"ARIMA(1,1,1)",20


## 4. Evaluation

Each model is fit on the pre-cutoff train segment and forecast over the full post-cutoff validation segment.

Important note: some forecast lines will still look straight even after removing the trivial baselines. That is not a plotting bug. Multi-step `SES`, `Rolling Mean`, and some small `ARIMA` models often collapse to level forecasts, especially when the training history is only 7 points.


In [5]:
results_rows = []
predictions = {}

for _, row in chosen.iterrows():
    series_df = get_series(row)
    train_part, val_part = split_series(series_df)
    y_train = train_part["y_target"].reset_index(drop=True)
    y_val = val_part["y_target"].to_numpy(dtype=float)
    w_val = val_part["weight"].to_numpy(dtype=float)
    steps = len(val_part)
    train_scale = float(np.mean(np.abs(np.diff(y_train)))) if len(y_train) > 1 else np.nan

    for model_name, spec in MODEL_SPECS.items():
        result = {
            "series_label": row["series_label"],
            "reason": row["reason"],
            "code": row["code"],
            "sub_code": row["sub_code"],
            "sub_category": row["sub_category"],
            "horizon": int(row["horizon"]),
            "model": model_name,
            "train_len": len(y_train),
            "val_len": steps,
            "status": "ok",
            "skip_reason": None,
            "skill_score": np.nan,
            "rmse": np.nan,
            "mae": np.nan,
            "mase": np.nan,
        }

        if len(y_train) < spec["min_train"]:
            result["status"] = "skipped"
            result["skip_reason"] = f"train_len<{spec['min_train']}"
            results_rows.append(result)
            continue

        try:
            pred = np.asarray(spec["func"](y_train, steps), dtype=float)
            if len(pred) != steps:
                raise ValueError("forecast_length_mismatch")
            if not np.all(np.isfinite(pred)):
                raise ValueError("non_finite_forecast")

            result["skill_score"] = weighted_skill(y_val, pred, w_val)
            result["rmse"] = weighted_rmse(y_val, pred, w_val)
            result["mae"] = weighted_mae(y_val, pred, w_val)
            result["mase"] = mase(y_val, pred, train_scale)
            predictions[(row["series_label"], model_name)] = pred
        except Exception as exc:
            result["status"] = "failed"
            result["skip_reason"] = type(exc).__name__

        results_rows.append(result)

results_df = pd.DataFrame(results_rows)
ok_results = results_df[results_df["status"] == "ok"].copy()
print(results_df["status"].value_counts())
display(results_df[["series_label", "model", "status", "skill_score", "rmse", "mae", "mase", "skip_reason"]].round(4))


status
ok         20
skipped     8
Name: count, dtype: int64


,series_label,model,status,skill_score,rmse,mae,mase,skip_reason
0,Longest History,Rolling Mean (w=10),ok,0.0000,3.0411,1.7411,1.5849,NaN
1,Longest History,SES,ok,0.0000,3.0448,1.6603,1.5102,NaN
2,Longest History,Holt,ok,0.0000,3.1438,2.0201,1.8240,NaN
3,Longest History,"ARIMA(1,0,0)",ok,0.0000,3.0479,1.6548,1.5052,NaN
4,Longest History,"ARIMA(0,1,1)",ok,0.0000,3.0443,1.6615,1.5113,NaN
5,Longest History,"ARIMA(1,1,0)",ok,0.0000,3.0714,1.8625,1.6961,NaN
6,Longest History,"ARIMA(1,1,1)",ok,0.0000,3.0472,1.6560,1.5063,NaN
7,Highest Total Weight,Rolling Mean (w=10),ok,0.0000,0.0006,0.0005,2.4856,NaN
8,Highest Total Weight,SES,ok,0.0000,0.0006,0.0005,2.4855,NaN
9,Highest Total Weight,Holt,ok,0.0000,0.0031,0.0028,12.3717,NaN


## 5. Tables and charts

This section keeps the comparison simple: per-series leaderboard, overall summary, skill heatmap, mean-skill chart, and fit-status chart.


In [6]:
ok_results["reason"] = pd.Categorical(ok_results["reason"], categories=REASON_ORDER, ordered=True)
ok_results = ok_results.sort_values(["reason", "skill_score", "rmse"], ascending=[True, False, True])

per_series_leaderboard = (
    ok_results[["series_label", "model", "skill_score", "rmse", "mae", "mase"]]
    .sort_values(["series_label", "skill_score", "rmse"], ascending=[True, False, True])
)

overall_summary = ok_results.groupby("model").agg(
    mean_skill=("skill_score", "mean"),
    mean_rmse=("rmse", "mean"),
    mean_mae=("mae", "mean"),
    mean_mase=("mase", "mean"),
    n_ok=("model", "size"),
)
overall_summary["n_attempted"] = results_df.groupby("model").size().reindex(overall_summary.index)
overall_summary["ok_rate"] = overall_summary["n_ok"] / overall_summary["n_attempted"]
overall_summary = overall_summary.sort_values(["mean_skill", "mean_rmse"], ascending=[False, True])

best_by_series = (
    ok_results.sort_values(["reason", "skill_score", "rmse"], ascending=[True, False, True])
    .groupby("series_label", as_index=False)
    .first()[["series_label", "model", "skill_score", "rmse", "mae", "mase"]]
)

print("Per-series best model:")
print(best_by_series.round(4).to_string(index=False))
print("\nDetailed leaderboard:")
display(per_series_leaderboard.round(4))
print("\nOverall model summary:")
display(overall_summary.round(4))

skill_pivot = ok_results.pivot(index="series_label", columns="model", values="skill_score").reindex(chosen["series_label"])
status_summary = results_df.groupby(["model", "status"]).size().unstack(fill_value=0).reindex(list(MODEL_SPECS), fill_value=0)
status_cols = [col for col in ["ok", "skipped", "failed"] if col in status_summary.columns]
status_colors = {"ok": "#55A868", "skipped": "#C44E52", "failed": "#8172B3"}

fig, axes = plt.subplots(1, 3, figsize=(21, 5), gridspec_kw={"width_ratios": [2.2, 1.0, 1.2]})

sns.heatmap(skill_pivot, annot=True, fmt=".3f", cmap="RdYlGn", center=0, ax=axes[0], cbar_kws={"label": "skill score"})
axes[0].set_title("Skill score by series x model")
axes[0].set_xlabel("model")
axes[0].set_ylabel("series")

mean_skill = overall_summary["mean_skill"].sort_values(ascending=True)
axes[1].barh(mean_skill.index, mean_skill.values, color="#4C72B0")
axes[1].set_title("Mean skill across the 4 series")
axes[1].set_xlabel("mean skill score")
axes[1].grid(True, axis="x", alpha=0.3)

status_summary[status_cols].plot(kind="bar", stacked=True, color=[status_colors[col] for col in status_cols], ax=axes[2])
axes[2].set_title("Fit status by model")
axes[2].set_xlabel("model")
axes[2].set_ylabel("count")
axes[2].tick_params(axis="x", rotation=45)
axes[2].legend(title="status", fontsize=8)

plt.tight_layout()
plt.show()


Per-series best model:
        series_label               model  skill_score     rmse      mae   mase
Highest Total Weight                 SES       0.0000   0.0006   0.0005 2.4855
     Longest History Rolling Mean (w=10)       0.0000   3.0411   1.7411 1.5849
         Most Stable                 SES       0.0000   0.0001   0.0001 0.3808
       Most Volatile        ARIMA(1,1,1)       0.8412 129.8104 108.8193 2.0917

Detailed leaderboard:


,series_label,model,skill_score,rmse,mae,mase
8,Highest Total Weight,SES,0.0000,0.0006,0.0005,2.4855
7,Highest Total Weight,Rolling Mean (w=10),0.0000,0.0006,0.0005,2.4856
9,Highest Total Weight,Holt,0.0000,0.0031,0.0028,12.3717
0,Longest History,Rolling Mean (w=10),0.0000,3.0411,1.7411,1.5849
4,Longest History,"ARIMA(0,1,1)",0.0000,3.0443,1.6615,1.5113
1,Longest History,SES,0.0000,3.0448,1.6603,1.5102
6,Longest History,"ARIMA(1,1,1)",0.0000,3.0472,1.6560,1.5063
3,Longest History,"ARIMA(1,0,0)",0.0000,3.0479,1.6548,1.5052
5,Longest History,"ARIMA(1,1,0)",0.0000,3.0714,1.8625,1.6961
2,Longest History,Holt,0.0000,3.1438,2.0201,1.8240



Overall model summary:


,mean_skill,mean_rmse,mean_mae,mean_mase,n_ok,n_attempted,ok_rate
model,,,,,,,
"ARIMA(1,1,1)",0.4206,66.4288,55.2377,1.7990,2,4,0.5
"ARIMA(1,1,0)",0.4131,69.1758,56.0577,1.9083,2,4,0.5
"ARIMA(0,1,1)",0.4120,69.5437,56.1673,1.8200,2,4,0.5
"ARIMA(1,0,0)",0.3678,82.8428,64.9147,1.9812,2,4,0.5
SES,0.2052,35.0584,28.2390,1.6293,4,4,1.0
Rolling Mean (w=10),0.1691,44.9619,35.5952,1.7899,4,4,1.0
Holt,0.1221,53.1606,41.3247,5.7751,4,4,1.0


## 6. Forecast comparison plots

Why do some forecast lines still look straight?

- `Rolling Mean` is constant by design
- `SES` usually becomes a flat level forecast for multi-step prediction
- some small ARIMA models also flatten over long horizons when there is no trend term
- two representative series only have 7 training points before the cutoff, so there is very little shape to learn

To keep the chart readable, the plot below shows the top unique forecast shapes per series rather than drawing multiple almost-identical flat lines on top of each other.


In [7]:
def select_unique_forecasts(series_label, top_k=3, atol=1e-8, rtol=1e-6):
    ranked = (
        ok_results[ok_results["series_label"] == series_label]
        .sort_values(["skill_score", "rmse"], ascending=[False, True])
    )
    selected = []
    seen_preds = []

    for _, model_row in ranked.iterrows():
        pred = predictions.get((series_label, model_row["model"]))
        if pred is None:
            continue
        if any(np.allclose(pred, prev, atol=atol, rtol=rtol) for prev in seen_preds):
            continue
        selected.append((model_row, pred))
        seen_preds.append(pred)
        if len(selected) == top_k:
            break

    return selected


fig, axes = plt.subplots(len(chosen), 1, figsize=(14, 12), sharex=False)
axes = np.atleast_1d(axes)
colors = ["#C44E52", "#8172B3", "#64B5CD"]

for ax, (_, row) in zip(axes, chosen.iterrows()):
    series_df = get_series(row)
    train_part, val_part = split_series(series_df)
    selected = select_unique_forecasts(row["series_label"], top_k=3)

    ax.plot(train_part["ts_index"], train_part["y_target"], color="black", linewidth=1.0, label="train")
    ax.plot(val_part["ts_index"], val_part["y_target"], color="#DD8452", linewidth=1.4, label="validation truth")

    for color, (model_row, pred) in zip(colors, selected):
        ax.plot(
            val_part["ts_index"],
            pred,
            color=color,
            linewidth=1.4,
            label=f"{model_row['model']} (skill={model_row['skill_score']:.3f})",
        )

    ax.axvline(VAL_CUTOFF, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{row['series_label']} - {row['code']} / {row['sub_code']} / H{row['horizon']}")
    ax.set_ylabel("y_target")
    ax.legend(loc="best", fontsize=8)

axes[-1].set_xlabel("ts_index")
plt.tight_layout()
plt.show()


## 7. Residuals and fit status

Residual histograms are shown only for the best valid model on each series.


In [8]:
display(status_summary)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for ax, (_, row) in zip(axes, best_by_series.iterrows()):
    chosen_row = chosen[chosen["series_label"] == row["series_label"]].iloc[0]
    series_df = get_series(chosen_row)
    _, val_part = split_series(series_df)
    pred = predictions[(row["series_label"], row["model"])]
    residual = val_part["y_target"].to_numpy(dtype=float) - pred

    ax.hist(residual, bins=25, color="#4C72B0", alpha=0.85)
    ax.axvline(0, color="red", linestyle="--", linewidth=1)
    ax.set_title(f"{row['series_label']} - best: {row['model']}")
    ax.set_xlabel("residual")
    ax.set_ylabel("count")

plt.tight_layout()
plt.show()


status,ok,skipped
model,,
Rolling Mean (w=10),4,0
SES,4,0
Holt,4,0
"ARIMA(1,0,0)",2,2
"ARIMA(0,1,1)",2,2
"ARIMA(1,1,0)",2,2
"ARIMA(1,1,1)",2,2


## 8. Final takeaways

This notebook is small on purpose, so the takeaway is direct: which simple classical model seems most useful for each representative series type.


In [9]:
final_summary = best_by_series.rename(columns={"model": "best_model"}).merge(
    results_df[["series_label", "train_len", "val_len"]].drop_duplicates(),
    on="series_label",
    how="left",
)
final_summary = final_summary.sort_values("series_label").reset_index(drop=True)

print("Best model for each representative series:")
print(final_summary.round(4).to_string(index=False))

print("\nShort reading:")
for _, row in final_summary.iterrows():
    print(
        f"- {row['series_label']}: {row['best_model']} "
        f"(skill={row['skill_score']:.3f}, rmse={row['rmse']:.3f}, train_len={int(row['train_len'])}, val_len={int(row['val_len'])})"
    )


Best model for each representative series:


        series_label          best_model  skill_score     rmse      mae   mase  train_len  val_len
Highest Total Weight                 SES       0.0000   0.0006   0.0005 2.4855          7      149
     Longest History Rolling Mean (w=10)       0.0000   3.0411   1.7411 1.5849         55      157
         Most Stable                 SES       0.0000   0.0001   0.0001 0.3808          7      163
       Most Volatile        ARIMA(1,1,1)       0.8412 129.8104 108.8193 2.0917        131       31

Short reading:
- Highest Total Weight: SES (skill=0.000, rmse=0.001, train_len=7, val_len=149)
- Longest History: Rolling Mean (w=10) (skill=0.000, rmse=3.041, train_len=55, val_len=157)
- Most Stable: SES (skill=0.000, rmse=0.000, train_len=7, val_len=163)
- Most Volatile: ARIMA(1,1,1) (skill=0.841, rmse=129.810, train_len=131, val_len=31)
